In [ ]:
!python -m pip install .. --quiet

In [ ]:
import ee 

ee.Authenticate() 
ee.Initialize(project='epistem2')

# 1. Satellite imagery

In [ ]:
# AOI definition

aoi = ee.FeatureCollection('projects/epistem2/assets/AOI_Sumatra').geometry()


In [ ]:

import geemap
from luma_ge.data_acquisition import Reflectance_Data, final_Image

#========== FIRST RETRIVE THE MULTISPECTRAL BAND===========
#Intialize the relfectance class data function
optical_reflectance = Reflectance_Data()
#Initialize the final image class for composite creation
composite = final_Image() #NEW FEATURE ADDED HERE
#define the start and end date for imagery collection
start = '2020-01-01'
end = '2020-12-31'
from luma_ge.data_acquisition import Reflectance_Data, final_Image

optical_reflectance = Reflectance_Data()

composite = final_Image() #NEW FEATURE ADDED HERE
#define the start and end date for imagery collection
start = '2020-01-01'
end = '2020-12-31'

landsat_data, meta = optical_reflectance.get_optical_data(
    aoi, start, end, optical_data='L8_SR', 
    cloud_cover=20,
    compute_detailed_stats=False  # skip expensive aggregations
)

landsat_data_clipped = landsat_data.map(lambda img: img.clip(aoi))

stacked_landsat = composite.get_quality_mosaic(
    landsat_data_clipped, aoi,
    calculate_coverage=False
)


# 2. Classification scheme 

In [ ]:
from luma_ge.classification_scheme import LULC_Scheme_Manager

manager = LULC_Scheme_Manager()

scheme_name = "Epistem"
success, message = manager.load_default_scheme(scheme_name)
classification_df = manager.get_dataframe()

print(classification_df.to_string(index=False))

# 3. Upload modular reference data

In [ ]:
import pandas as pd
import numpy as np
import json

# use the modular reference dataset was reverse engineered from KLHK map
df_train_csv = '../data/modular_mapping_approach/sumatra_test/sumatra_extracted_raster_values_by_provinces_CLEANED.csv'
df_train = pd.read_csv(df_train_csv)

# Extract lon and lat
# df_train[['longitude', 'latitude']] = (
#     df_train['geometry']
#     .str.extract(r'POINT\s*\(([-\d.]+)\s+([-\d.]+)\)')
#     .astype(float)
# )

# df_train = df_train.drop(columns=['geometry'])

df_train[['longitude', 'latitude']] = (
    df_train['.geo']
    .apply(lambda x: json.loads(x)['coordinates'])
    .apply(pd.Series)
)

df_train = df_train.drop(columns=['.geo', 'system:index'])

print(df_train.head())

In [ ]:
from luma_ge.classification_scheme import LULC_Scheme_Manager

manager = LULC_Scheme_Manager()

scheme_name = "Epistem"
success, message = manager.load_default_scheme(scheme_name)
classification_df = manager.get_dataframe()

print(classification_df.to_string(index=False))

## Define default scheme labelling ruleset

In [ ]:
# ── DEFINE RULESET ─────────────────────────────────────────────────────────
# Each row = one class rule. Columns are primitives with operators.
# Format: ">0.40" means "greater than 0.40"
#         "==1" means "equal to 1"
#         ">=25" means "greater than or equal to 25"
#         'treecover': '<30 | >80',  means treecover < 30 OR treecover > 80   
#         None means "no condition on this primitive"

ruleset_csv = '../data/modular_mapping_approach/ruleset_epistem_default_v5.csv'
ruleset = pd.read_csv(ruleset_csv)
ruleset = ruleset.sort_values(by='priority', ascending=True)
print(ruleset)

# copy to clipboard for easy pasting into the ruleset CSV file
pd.DataFrame.to_clipboard(ruleset)

CLASS_NAMES = dict(zip(ruleset['class_id'], ruleset['class_name']))
CLASS_NAMES[0] = 'unclassified'
CLASS_NAMES[-1] = 'abstain'

## Helper functions to label the classes

In [ ]:
def safe_num(val, default=0):
    """Convert value to float, return default if None or NaN."""
    if val is None:
        return default
    if isinstance(val, (int, float)):
        if np.isnan(val):
            return default
        return float(val)
    try:
        return float(val)
    except:
        return default

def evaluate_condition(row_val, condition_str):
    """
    Evaluate a single condition: row_val op threshold?
    Supports OR logic with pipe separator: ">0.40|<0.10"
    
    Args:
        row_val: The value from the sample
        condition_str: String like ">0.40", ">=25", ">0.40|<0.10"
    
    Returns:
        bool: True if condition is satisfied, False otherwise
    """
    if condition_str is None:
        return True  # No condition → always passes
    
    condition_str = str(condition_str).strip()
    
    # Handle OR logic (pipe-separated conditions)
    if '|' in condition_str:
        sub_conditions = [c.strip() for c in condition_str.split('|')]
        return any(evaluate_condition(row_val, c) for c in sub_conditions)

    # Handle AND logic
    if '&' in condition_str:
        sub_conditions = [c.strip() for c in condition_str.split('&')]
        return all(evaluate_condition(row_val, c) for c in sub_conditions)
    
    # Parse operator and threshold
    if condition_str.startswith('=='):
        op, threshold_str = '==', condition_str[2:]
    elif condition_str.startswith('>='):
        op, threshold_str = '>=', condition_str[2:]
    elif condition_str.startswith('<='):
        op, threshold_str = '<=', condition_str[2:]
    elif condition_str.startswith('>'):
        op, threshold_str = '>', condition_str[1:]
    elif condition_str.startswith('<'):
        op, threshold_str = '<', condition_str[1:]
    else:
        return True  # Invalid condition → pass
    
    threshold = safe_num(threshold_str)
    row_val_num = safe_num(row_val)
    
    if op == '>':
        return row_val_num > threshold
    elif op == '>=':
        return row_val_num >= threshold
    elif op == '<':
        return row_val_num < threshold
    elif op == '<=':
        return row_val_num <= threshold
    elif op == '==':
        return np.isclose(
            row_val_num,
            threshold,
            atol=1e-6
        )
    
    return False

def check_rule_match(row, rule):
    """
    Check if a sample row matches all conditions in a rule.
    
    Args:
        row: pd.Series with sample data
        rule: pd.Series with rule conditions
    
    Returns:
        bool: True if ALL conditions are satisfied
    """
    # Get all primitive columns (skip metadata like class_id, class_name, priority)
    metadata = {'class_id', 'class_name', 'priority'}
    primitive_cols = [col for col in rule.index if col not in metadata]
    
    for prim in primitive_cols:
        condition = rule[prim]
        
        # Handle special case for range checks
        if pd.isna(condition) or condition is None:
            continue  # No condition on this primitive
        
        row_val = row.get(prim, np.nan)
        
        if not evaluate_condition(row_val, condition):
            return False  # Any condition fails → rule doesn't match
    
    return True  # All conditions passed

print('✓ Helper functions defined')

## Assign the class labels to the modular reference data

Using previous labels to narrow down the potential labels in the new classification scheme

In [ ]:
def assign_label(row, ruleset):
    """
    Evaluate rules in priority order.

    - First matching rule returns its class_id.
    - Non-matching rules abstain (-1) and evaluation continues.
    - If every rule abstains, return 0 (unclassified).

    """
    ruleset_sorted = ruleset.sort_values("priority").reset_index(drop=True)

    for _, rule in ruleset_sorted.iterrows():
         if check_rule_match(row, rule):
            return rule["class_id"]  # first vote wins

    return 0  # all LFs abstained

df_train['label'] = df_train.apply(lambda row: assign_label(row, ruleset), axis=1)

# add class_name column
class_mapping = ruleset.set_index('class_id')['class_name'].to_dict()

df_train['class_name'] = df_train['label'].map(class_mapping).fillna('unclassified')

print(f'✓ Labels assigned to {len(df_train)} samples\n')

print('Label distribution (assigned):')
print(
    df_train['class_name']
    .value_counts()
    .reindex(ruleset['class_name'].unique(), fill_value=0)
)

print(f'\nUnclassified (label=0): {(df_train["label"] == 0).sum()}')

## Compare with original class label

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ORIGINAL_CLASS = "LULC_24"

print(f"Classified samples: {len(df_train)}")
print()


# Complete list of target classes + unclassified
all_classes = list(ruleset["class_name"].unique()) + ["unclassified"]

# Count how many samples from each original class received each label
summary = (
    pd.crosstab(
        df_train[ORIGINAL_CLASS],
        df_train["class_name"]
    )
    .reindex(columns=all_classes, fill_value=0)
)

print("Original class -> Assigned labels")
print(summary)

summary.to_clipboard(
    excel=True,
    index=True
)

## Save the labelled training dataset

In [ ]:
import geopandas as gpd
from shapely.geometry import Point

geometry = [Point(xy) for xy in zip(df_train["longitude"], df_train["latitude"])]
gdf = gpd.GeoDataFrame(df_train, geometry=geometry)
gdf.to_file("../data/modular_mapping_approach/sumatra_test/sumatra_td_DTL_result_v5_cleaned.shp")

## Sanitize relabelled shp file with original Luma module 3 script

In [ ]:
from luma_ge.sample_data import SyncTrainData

LULCTable = classification_df
TrainVectPath = "../data/modular_mapping_approach/sumatra_test/sumatra_td_DTL_result_v5_cleaned.shp"

TrainField = 'labels' 
        # Load and process training data
TrainDataDict = SyncTrainData.LoadTrainData(
            landcover_df=LULCTable,
            aoi_geometry=aoi,
            training_shp_path=TrainVectPath
        )

In [ ]:
# ----- System response 3.2.a -----
# Set class field
TrainDataDict = SyncTrainData.SetClassField(TrainDataDict, TrainField)

# Validate classes
TrainDataDict = SyncTrainData.ValidClass(TrainDataDict,1)

    # Check sample sufficiency
TrainDataDict = SyncTrainData.CheckSufficiency(TrainDataDict, min_samples=20)

    # Filter by AOI
TrainDataDict = SyncTrainData.FilterTrainAoi(TrainDataDict)

    # Create training data table
table_df, total_samples, insufficient_df = SyncTrainData.TrainDataRaw(
    training_data=TrainDataDict.get('training_data'),
    landcover_df=TrainDataDict.get('landcover_df'),
    class_field=TrainDataDict.get('class_field'))

#Summary result
vr = TrainDataDict.get('validation_results', {})

print("=" * 70)
print("TRAINING DATA SUMMARY")
print("=" * 70)
print(f"Total training points loaded     : {vr.get('total_points', 'N/A')}")
print(f"Points after class filtering     : {vr.get('points_after_class_filter', 'N/A')}")
print(f"Valid points (inside AOI)        : {vr.get('valid_points', 'N/A')}")
print(f"Invalid classes found            : {len(vr.get('invalid_classes', []))}")
print(f"Points outside AOI               : {len(vr.get('outside_aoi', []))}")
print("=" * 70)

    # --- Display the main table ---
if table_df is not None and not table_df.empty:
        display_df = table_df.copy()
        if 'Percentage' in display_df.columns:
            display_df['Percentage'] = display_df['Percentage'].apply(
                lambda x: f"{x:.2f}%" if isinstance(x, (int, float)) else x
            )
        display(display_df)
else:
        print("No valid training data available to display.")

TrainDataFinal = TrainDataDict.get('training_data')

# 6. Land cover classification

## Generate multiprobability classification map

In [ ]:
from ee import classifier

from luma_ge.classification import FeatureExtraction

labeled_roi = geemap.gdf_to_ee(TrainDataFinal)

#Perform Training Test Split
features = FeatureExtraction()
strafied_train, stratified_test = features.stratified_split(labeled_roi, stacked_landsat, 
                            class_prop='ID', train_ratio=1)

# create classifier in multiprobability output mode
clf_sumatra_prob = ee.Classifier.smileRandomForest(
    numberOfTrees=30,
    minLeafPopulation=1
).setOutputMode('MULTIPROBABILITY').train(
        features=strafied_train,
        classProperty='ID',
        inputProperties=stacked_landsat.bandNames()
        )

In [ ]:
# rename band names of the probability
import re


def sanitize_band_name(name: str) -> str:
    """GEE band names should avoid spaces/special chars for safety downstream."""
    name = str(name).strip()
    name = re.sub(r'[^\w]+', '_', name)   # replace non-word chars with underscore
    name = re.sub(r'_+', '_', name).strip('_')
    return name

classification_df = classification_df.sort_values("ID").reset_index(drop=True)
class_labels = [sanitize_band_name(name) for name in classification_df["Land Cover Class"]]

#classify probability
probability_stack = stacked_landsat.classify(clf_sumatra_prob)
probability_stack = probability_stack.arrayFlatten(
    [class_labels]
)

# # Check the final GEE band names
# print("\nFinal probability stack band names:")
# print(probability_stack.bandNames().getInfo())

# 7. Export Multiprobability stack

Create loop per provinces because exporting them all at once per island keeps failing

In [ ]:
# Province boundaries
provinces = ee.FeatureCollection('projects/epistem2/assets/AOI_Sumatra_Provinces')

# Get province names and geometries
province_list = provinces.toList(provinces.size())

n_provinces = provinces.size().getInfo()

for i in range(n_provinces):

    province = ee.Feature(province_list.get(i))

    # Change 'AoI' to the actual province-name field
    province_name = province.get('AoI').getInfo()

    province_geom = province.geometry()

    # Clean province name for use in task/file names
    province_name_clean = (
        province_name
        .replace(' ', '_')
        .replace('/', '_')
        .replace('-', '_')
    )

    print(f"Creating export for: {province_name}")

    task = ee.batch.Export.image.toDrive(
        image=probability_stack,
        description=f'sumatra_probability_{province_name_clean}_Epistem_v5',
        folder='sumatra_multiprobability_stack_Epistem',
        fileNamePrefix=f'sumatra_probability_{province_name_clean}_Epistem_v5',
        region=province_geom,
        scale=100,
        # crs='EPSG:4326',
        maxPixels=1e13,
        shardSize=8,
        fileFormat='GeoTIFF',
        formatOptions={
            'cloudOptimized': True
        }
    )

    task.start()

    print(f"  ✓ Started: {province_name_clean}")